# Seq2Seq with Bahdanau Attention for Hindi Transliteration

PyTorch encoder–decoder with additive (Bahdanau) attention, trained on the Dakshina Hindi corpus.
Input: Latin/Roman transliteration. Output: Devanagari Hindi.

Set the `DAKSHINA_DATA_DIR` environment variable (or edit `DATA_DIR` below) before running.

In [ ]:
import os
import math
import random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Data Loading

In [ ]:
from google.colab import files
import io

print('Upload 3 TSV files: train, dev, and test')

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
uploaded = files.upload()

def load_tsv_pairs(content):
    pairs = []
    for line in io.StringIO(content.decode('utf-8')):
        parts = line.strip().split('\t')
        if len(parts) < 3:
            continue
        native = parts[0].strip()
        roman = parts[1].strip()
        if roman and native:
            pairs.append((roman, native))
    if not pairs:
        raise ValueError('No data loaded from file')
    return pairs

train_pairs = None
dev_pairs = None
test_pairs = None

for filename, content in uploaded.items():
    if 'train' in filename.lower():
        train_pairs = load_tsv_pairs(content)
    elif 'dev' in filename.lower() or 'valid' in filename.lower():
        dev_pairs = load_tsv_pairs(content)
    elif 'test' in filename.lower():
        test_pairs = load_tsv_pairs(content)
    else:
        if train_pairs is None:
            train_pairs = load_tsv_pairs(content)
        elif dev_pairs is None:
            dev_pairs = load_tsv_pairs(content)
        elif test_pairs is None:
            test_pairs = load_tsv_pairs(content)

assert train_pairs is not None and dev_pairs is not None and test_pairs is not None,     'Could not identify all three splits. Make sure filenames contain train/dev/test.'

print(f'Train: {len(train_pairs)}  Dev: {len(dev_pairs)}  Test: {len(test_pairs)}')
print('Sample:', train_pairs[:3])


## Vocabulary and Dataset

In [ ]:
def build_vocab(pairs):
    src_chars, tgt_chars = set(), set()
    for s, t in pairs:
        src_chars.update(s)
        tgt_chars.update(t)
    tokens = SPECIAL_TOKENS + sorted(src_chars)
    src2idx = {c: i for i, c in enumerate(tokens)}
    tokens = SPECIAL_TOKENS + sorted(tgt_chars)
    tgt2idx = {c: i for i, c in enumerate(tokens)}
    idx2tgt = {i: c for c, i in tgt2idx.items()}
    return src2idx, tgt2idx, idx2tgt


src2idx, tgt2idx, idx2tgt = build_vocab(train_pairs + dev_pairs)
PAD, SOS, EOS, UNK = (
    tgt2idx["<PAD>"], tgt2idx["<SOS>"],
    tgt2idx["<EOS>"], tgt2idx["<UNK>"]
)
SRC_VOCAB_SIZE = len(src2idx)
TGT_VOCAB_SIZE = len(tgt2idx)
MAX_LEN = 32

print(f"Source vocab: {SRC_VOCAB_SIZE}  Target vocab: {TGT_VOCAB_SIZE}")


In [ ]:
class TransliterationDataset(Dataset):
    def __init__(self, pairs, max_len=MAX_LEN):
        self.pairs = pairs
        self.max_len = max_len

    def encode(self, text, vocab, add_special=True):
        ids = [vocab.get(c, UNK) for c in text[: self.max_len - 2]]
        if add_special:
            ids = [SOS] + ids + [EOS]
        else:
            ids = ids + [EOS]
        return torch.LongTensor(ids)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return (
            self.encode(src, src2idx, add_special=False),
            self.encode(tgt, tgt2idx, add_special=True),
        )


def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_max = max(len(s) for s in srcs)
    tgt_max = max(len(t) for t in tgts)
    src_pad = torch.full((len(srcs), src_max), PAD, dtype=torch.long)
    tgt_pad = torch.full((len(tgts), tgt_max), PAD, dtype=torch.long)
    src_lens = torch.LongTensor([len(s) for s in srcs])
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, : len(s)] = s
        tgt_pad[i, : len(t)] = t
    return src_pad, tgt_pad, src_lens


BATCH_SIZE = 128
train_loader = DataLoader(TransliterationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(TransliterationDataset(dev_pairs), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(TransliterationDataset(test_pairs), batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)}")


## Model: BiRNN Encoder + Bahdanau Attention Decoder

- **Encoder** — bidirectional RNN (`lstm`/`gru`/`rnn`), directions fused by summation so `enc_dim == hidden_dim`.
- **Decoder** — unidirectional RNN; at each step Bahdanau additive attention
  `score(h_t, e_i) = v^T tanh(W_enc e_i + U_dec h_t)` attends over encoder outputs.

In [ ]:
class Encoder(nn.Module):
    """Bidirectional RNN encoder (BiRNN).

    With bidirectional=True each direction has `hidden_dim` units; the two
    per-direction output halves are summed so enc_outputs have dim `hidden_dim`,
    which is what the attention module expects as enc_dim.
    """

    def __init__(self, vocab_size, emb_dim=64, hidden_dim=256,
                 num_layers=1, cell_type="lstm", bidirectional=True, dropout=0.1):
        super().__init__()
        self.cell_type = cell_type.lower()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        rnn_cls = {"lstm": nn.LSTM, "gru": nn.GRU, "rnn": nn.RNN}[self.cell_type]
        self.rnn = rnn_cls(
            emb_dim, hidden_dim, num_layers,
            batch_first=True, bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0
        )
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, src, src_lens):
        embedded = self.embedding(src)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, src_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        outputs_packed, hidden = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            outputs_packed, batch_first=True
        )
        if self.bidirectional:
            # Fuse forward/backward halves: (B, T, 2*H) -> (B, T, H)
            outputs = outputs[:, :, : self.hidden_dim] + outputs[:, :, self.hidden_dim:]
        return outputs, hidden


class BahdanauAttention(nn.Module):
    """Additive attention: score(h_t, e_i) = v^T tanh(W_enc e_i + U_dec h_t)."""

    def __init__(self, enc_dim, dec_dim, attn_dim=128):
        super().__init__()
        self.W_enc = nn.Linear(enc_dim, attn_dim, bias=False)
        self.U_dec = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, enc_outputs, dec_hidden, src_mask):
        # enc_outputs: (batch, src_len, enc_dim)
        # dec_hidden: (batch, dec_dim)
        enc_proj = self.W_enc(enc_outputs)                    # (batch, src_len, attn_dim)
        dec_proj = self.U_dec(dec_hidden).unsqueeze(1)       # (batch, 1, attn_dim)
        scores = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(-1)  # (batch, src_len)
        scores = scores.masked_fill(src_mask, -1e9)
        attn_weights = F.softmax(scores, dim=1)               # (batch, src_len)
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)   # (batch, 1, enc_dim)
        return context.squeeze(1), attn_weights               # (batch, enc_dim), (batch, src_len)


class Decoder(nn.Module):
    """Unidirectional RNN decoder with Bahdanau additive attention."""

    def __init__(self, vocab_size, emb_dim=64, hidden_dim=256,
                 enc_dim=256, num_layers=1, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.cell_type = cell_type.lower()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        rnn_cls = {"lstm": nn.LSTM, "gru": nn.GRU, "rnn": nn.RNN}[self.cell_type]
        self.rnn = rnn_cls(
            emb_dim + enc_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.attention = BahdanauAttention(enc_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim + enc_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, enc_outputs, src_mask):
        # input_tok: (batch,)
        # hidden: tuple (h, c) for LSTM, tensor for GRU/RNN
        dec_hidden = hidden[0][-1] if isinstance(hidden, tuple) else hidden[-1]

        context, attn_weights = self.attention(enc_outputs, dec_hidden, src_mask)
        emb = self.embedding(input_tok).unsqueeze(1)  # (batch, 1, emb_dim)
        rnn_input = torch.cat([emb, context.unsqueeze(1)], dim=2)  # (batch, 1, emb+enc)
        output, hidden = self.rnn(rnn_input, hidden)
        output = output.squeeze(1)  # (batch, hidden_dim)
        logits = self.fc_out(self.dropout(torch.cat([output, context], dim=1)))
        return logits, hidden, attn_weights


class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx=PAD):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx

    def _init_decoder_hidden(self, enc_hidden):
        """Map encoder hidden state (num_layers*num_dirs, B, hidden) to
        decoder init state (num_layers, B, hidden) by summing directions."""
        num_layers = self.encoder.num_layers
        if self.encoder.bidirectional:
            if isinstance(enc_hidden, tuple):  # LSTM: (h, c)
                h, c = enc_hidden
                h = h.view(num_layers, 2, h.size(1), -1).sum(dim=1)
                c = c.view(num_layers, 2, c.size(1), -1).sum(dim=1)
                return (h, c)
            h = enc_hidden.view(num_layers, 2, enc_hidden.size(1), -1).sum(dim=1)
            return h
        return enc_hidden

    def forward(self, src, tgt, src_lens, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        src_mask = (src == self.src_pad_idx)  # (batch, src_len) True where padded

        enc_outputs, enc_hidden = self.encoder(src, src_lens)
        dec_hidden = self._init_decoder_hidden(enc_hidden)

        outputs = torch.zeros(batch_size, tgt_len, TGT_VOCAB_SIZE, device=src.device)
        input_tok = tgt[:, 0]  # <SOS>

        for t in range(1, tgt_len):
            logits, dec_hidden, _ = self.decoder(input_tok, dec_hidden, enc_outputs, src_mask)
            outputs[:, t, :] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = logits.argmax(1)
            input_tok = tgt[:, t] if teacher_force else top1

        return outputs


EMB_DIM = 64
HIDDEN_DIM = 256
NUM_LAYERS = 1
CELL_TYPE = "lstm"
BIDIRECTIONAL = True   # BiRNN encoder
DROPOUT = 0.1
TEACHER_FORCING = 0.5
LR = 3e-4
EPOCHS = 20

enc = Encoder(SRC_VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS,
              cell_type=CELL_TYPE, bidirectional=BIDIRECTIONAL, dropout=DROPOUT)
dec = Decoder(TGT_VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, enc_dim=HIDDEN_DIM,
              num_layers=NUM_LAYERS, cell_type=CELL_TYPE, dropout=DROPOUT)
model = Seq2SeqAttention(enc, dec).to(DEVICE)

optimizer = Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

#### Hyperparameter Sweep (W&B) — Telugu

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val_bleu", "goal": "maximize"},
    "parameters": {
        "emb_dim": {"values": [32, 64, 128]},
        "hidden_dim": {"values": [64, 128, 256]},
        "num_layers": {"values": [1, 2]},
        "cell_type": {"values": ["lstm", "gru"]},
        "dropout": {"values": [0.1, 0.2, 0.3]},
        "attn_dim": {"values": [64, 128]},
        "learning_rate": {"min": 1e-4, "max": 5e-3},
        "batch_size": {"values": [32, 64, 128]},
        "teacher_forcing_ratio": {"values": [0.5, 0.7, 1.0]},
    }
}

## Training Loop

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0.0
    for src, tgt, src_lens in tqdm(loader, desc="Training", leave=False):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        src_lens = src_lens.to(DEVICE)
        optimizer.zero_grad()
        output = model(src, tgt, src_lens, teacher_forcing_ratio=TEACHER_FORCING)
        loss = criterion(
            output[:, 1:].reshape(-1, TGT_VOCAB_SIZE),
            tgt[:, 1:].reshape(-1)
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss = 0.0
    for src, tgt, src_lens in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        src_lens = src_lens.to(DEVICE)
        output = model(src, tgt, src_lens, teacher_forcing_ratio=0.0)
        loss = criterion(
            output[:, 1:].reshape(-1, TGT_VOCAB_SIZE),
            tgt[:, 1:].reshape(-1)
        )
        total_loss += loss.item()
    return total_loss / len(loader)


best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(train_loader)
    val_loss = evaluate_loss(dev_loader)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "attention_best.pt")
        print("  Saved best model.")


#### Beam Search Decoding & Attention Heatmaps

In [ ]:
import heapq

@torch.no_grad()
def beam_search_decode(model, src_text, beam_size=4, max_len=32):
    """Beam search for Bahdanau attention model."""
    model.eval()
    src_ids = [src2idx.get(c, UNK) for c in src_text]
    src_tensor = torch.LongTensor(src_ids + [EOS]).unsqueeze(0).to(DEVICE)
    src_lens = torch.LongTensor([len(src_ids) + 1]).to(DEVICE)
    src_mask = (src_tensor == PAD)

    enc_outputs, enc_hidden = model.encoder(src_tensor, src_lens)
    dec_hidden = model._init_decoder_hidden(enc_hidden)

    # beam: list of (log_prob, token_ids, hidden_state)
    beam = [(0.0, [SOS], dec_hidden)]
    completed = []

    for _ in range(max_len):
        candidates = []
        for score, tokens, hidden in beam:
            if tokens[-1] == EOS:
                completed.append((score, tokens))
                continue
            inp = torch.LongTensor([tokens[-1]]).to(DEVICE)
            logits, new_hidden, _ = model.decoder(inp, hidden, enc_outputs, src_mask)
            log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)
            topk_logp, topk_idx = torch.topk(log_probs, beam_size)
            for k in range(beam_size):
                new_score = score + topk_logp[k].item()
                new_tokens = tokens + [topk_idx[k].item()]
                # clone hidden for LSTM/GRU
                if isinstance(new_hidden, tuple):
                    h, c = new_hidden
                    new_h = (h.clone(), c.clone())
                else:
                    new_h = new_hidden.clone()
                candidates.append((new_score, new_tokens, new_h))
        # keep top beam_size
        beam = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]
        if len(completed) >= beam_size:
            break

    if not completed:
        completed = beam
    best_tokens = max(completed, key=lambda x: x[0])[1]
    # strip SOS/EOS/PAD
    out = []
    for tok in best_tokens:
        if tok == SOS or tok == PAD:
            continue
        if tok == EOS:
            break
        out.append(idx2tgt.get(tok, "?"))
    return "".join(out)

#### Visualize Attention

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def plot_attention_heatmap(model, src_text, figsize=(8, 4)):
    """Visualize Bahdanau attention weights for a word."""
    model.eval()
    src_ids = [src2idx.get(c, UNK) for c in src_text]
    src_tensor = torch.LongTensor(src_ids + [EOS]).unsqueeze(0).to(DEVICE)
    src_lens = torch.LongTensor([len(src_ids) + 1]).to(DEVICE)
    src_mask = (src_tensor == PAD)

    enc_outputs, enc_hidden = model.encoder(src_tensor, src_lens)
    dec_hidden = model._init_decoder_hidden(enc_hidden)

    ys = torch.LongTensor([SOS]).to(DEVICE)
    attn_mat = []
    decoded = []
    for _ in range(30):
        logits, dec_hidden, attn_w = model.decoder(ys, dec_hidden, enc_outputs, src_mask)
        attn_mat.append(attn_w.squeeze(0).cpu().tolist())
        nxt = logits.argmax(-1).item()
        if nxt == EOS:
            break
        decoded.append(idx2tgt.get(nxt, "?"))
        ys = torch.LongTensor([nxt]).to(DEVICE)

    if not attn_mat:
        print("No attention weights collected")
        return
    import numpy as np
    attn_mat = np.array(attn_mat)  # (tgt_len, src_len)
    src_chars = list(src_text) + ["<EOS>"]
    tgt_chars = decoded

    plt.figure(figsize=figsize)
    plt.imshow(attn_mat, aspect="auto", cmap="viridis")
    plt.xticks(range(len(src_chars)), src_chars, rotation=45)
    plt.yticks(range(len(tgt_chars)), tgt_chars)
    plt.xlabel("Source (Roman)")
    plt.ylabel("Target (Telugu) — greedy steps")
    plt.title(f"Bahdanau attention: '{src_text}' → '{''.join(decoded)}'")
    plt.colorbar(label="Attention weight")
    plt.tight_layout()
    plt.show()
    return attn_mat

## Greedy Inference and Evaluation

In [ ]:
@torch.no_grad()
def translate_greedy(src_text, max_len=MAX_LEN):
    model.eval()
    src_ids = [src2idx.get(c, UNK) for c in src_text]
    src_tensor = torch.LongTensor(src_ids + [EOS]).unsqueeze(0).to(DEVICE)
    src_lens = torch.LongTensor([len(src_ids) + 1]).to(DEVICE)
    src_mask = (src_tensor == PAD)

    enc_outputs, enc_hidden = model.encoder(src_tensor, src_lens)
    dec_hidden = model._init_decoder_hidden(enc_hidden)
    ys = torch.LongTensor([SOS]).to(DEVICE)

    decoded_ids = []
    for _ in range(max_len - 1):
        logits, dec_hidden, _ = model.decoder(ys, dec_hidden, enc_outputs, src_mask)
        next_token = logits.argmax(-1).item()
        if next_token == EOS:
            break
        decoded_ids.append(next_token)
        ys = torch.LongTensor([next_token]).to(DEVICE)

    return "".join(idx2tgt[i] for i in decoded_ids)


@torch.no_grad()
def evaluate_test(loader):
    model.eval()
    correct_words = total_words = correct_chars = total_chars = 0
    for src_batch, tgt_batch, src_lens in tqdm(loader, desc="Testing"):
        src_batch = src_batch.to(DEVICE)
        for i in range(src_batch.size(0)):
            src_ids = src_batch[i][src_batch[i] != PAD].tolist()
            src_text = "".join(
                next((c for c, idx in src2idx.items() if idx == j), "")
                for j in src_ids[:-1]
            )
            pred = translate_greedy(src_text)

            true_ids = [
                j.item() for j in tgt_batch[i]
                if j.item() not in (PAD, SOS, EOS)
            ]
            truth = "".join(idx2tgt[j] for j in true_ids)

            if pred == truth:
                correct_words += 1
            total_words += 1
            min_len = min(len(pred), len(truth))
            correct_chars += sum(a == b for a, b in zip(pred, truth[:min_len]))
            total_chars += len(truth)

    word_acc = correct_words / total_words if total_words else 0
    char_acc = correct_chars / total_chars if total_chars else 0
    print(f"Word Accuracy: {word_acc:.4f} | Char Accuracy: {char_acc:.4f}")
    return {"word_accuracy": word_acc, "char_accuracy": char_acc}


# Load best model and evaluate
model.load_state_dict(torch.load("attention_best.pt", map_location=DEVICE))
metrics = evaluate_test(test_loader)

# Show sample predictions
for roman, deva in test_pairs[:10]:
    print(f"{roman:>15s} -> {translate_greedy(roman):<15s} (truth: {deva})")
